In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "20"
os.environ["MKL_NUM_THREADS"] = "20"

import torch
torch.set_num_threads(20)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
import random
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models.video import r3d_18

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
class VideoDataset(Dataset):

    def __init__(self, root, frames=16, spatial_size=112):

        self.root = Path(root)
        self.frames = frames
        self.spatial_size = spatial_size

        self.samples = []

        normal_folder = self.root / "normal"
        crime_folder = self.root / "crime"

        exts = (".mp4",".avi",".mov",".mkv")

        for p in normal_folder.rglob("*"):
            if p.suffix.lower() in exts:
                self.samples.append((str(p),0))

        for p in crime_folder.rglob("*"):
            if p.suffix.lower() in exts:
                self.samples.append((str(p),1))


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):

        path, label = self.samples[idx]

        cap = cv2.VideoCapture(path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if frame_count <= 0:
            frame_count = self.frames

        indices = np.linspace(0, frame_count-1, self.frames).astype(int)

        frames = []

        for i in indices:

            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ret, frame = cap.read()

            if not ret:
                frame = np.zeros((self.spatial_size,self.spatial_size,3),dtype=np.uint8)

            frame = cv2.resize(frame,(self.spatial_size,self.spatial_size))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frame = torch.tensor(frame).permute(2,0,1).float()/255.0
            frames.append(frame)

        cap.release()

        clip = torch.stack(frames)
        clip = clip.permute(1,0,2,3)

        return clip, torch.tensor(label).float()

In [5]:
TRAIN_ROOT = r"D:\Dtaset fot cctv\UCF"

frames = 32
spatial = 128
batch_size = 4
num_workers = 0

dataset = VideoDataset(TRAIN_ROOT, frames=frames, spatial_size=spatial)

print("Total videos:",len(dataset))

crime = sum(1 for _,l in dataset.samples if l==1)
normal = sum(1 for _,l in dataset.samples if l==0)

print("Crime videos:",crime)
print("Normal videos:",normal)

Total videos: 1900
Crime videos: 950
Normal videos: 950


In [6]:
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size

train_ds,val_ds = random_split(dataset,[train_size,val_size])

train_ds.dataset.mode="train"
val_ds.dataset.mode="val"

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

print("Train:",len(train_ds),"Val:",len(val_ds))

Train: 1520 Val: 380


In [7]:
model = r3d_18(pretrained=True)

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features,1)
)

model = model.to(device)

print(model)

C:\Users\amarh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\amarh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VideoResNet(
  (stem): BasicStem(
    (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1):

In [8]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=3e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=2,
    factor=0.5
)

In [9]:
EPOCHS = 12

for epoch in range(EPOCHS):

    model.train()

    train_correct = 0
    train_total = 0
    running_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for clips, labels in pbar:

        clips = clips.to(device)
        labels = labels.to(device)

        logits = model(clips).squeeze(1)

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.4).float()

        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        pbar.set_postfix(
            loss = running_loss/train_total,
            train_acc = train_correct/train_total
        )

    train_acc = train_correct / train_total


    # ---------- VALIDATION ----------

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for clips, labels in val_loader:

            clips = clips.to(device)
            labels = labels.to(device)

            logits = model(clips).squeeze(1)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch+1} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    scheduler.step(1 - val_acc)

    torch.save(model.state_dict(), f"s3_r3d_epoch_{epoch+1}.pth")

Epoch 1/12: 100%|██████████| 380/380 [1:31:51<00:00, 14.50s/it, loss=0.517, train_acc=0.733]



Epoch 1 | Train Acc: 0.7329 | Val Acc: 0.8500


Epoch 2/12: 100%|██████████| 380/380 [1:38:28<00:00, 15.55s/it, loss=0.344, train_acc=0.848]



Epoch 2 | Train Acc: 0.8480 | Val Acc: 0.8605


Epoch 3/12: 100%|██████████| 380/380 [2:07:58<00:00, 20.21s/it, loss=0.29, train_acc=0.866]   


...


Epoch 9 | Train Acc: 0.9375 | Val Acc: 0.8974


Epoch 10/12: 100%|██████████| 380/380 [1:37:14<00:00, 15.35s/it, loss=0.157, train_acc=0.934]



Epoch 10 | Train Acc: 0.9336 | Val Acc: 0.8447


Epoch 11/12:   8%|▊         | 31/380 [11:52<2:13:37, 22.97s/it, loss=0.228, train_acc=0.895]


KeyboardInterrupt: 